# ROS04 · ROS 2 actions and launch files

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS04_ros_2_actions_and_launch_files.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 60 min

**Skill Hub concepts:** ROS 2 actions; ROS 2 launch files

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS04")

## Map: an action is three services and two topics

```
 client                                   server
   ──── send_goal (service) ──────────────▶ goal_callback: ACCEPT or REJECT
   ◀─── feedback (topic) ──────────────────  execute_callback publishes progress
   ──── cancel_goal (service) ────────────▶ cancel_callback: ACCEPT or REJECT; execute checks is_cancel_requested
   ◀─── status (topic) ────────────────────  ACCEPTED, EXECUTING, CANCELING, SUCCEEDED, CANCELED, ABORTED
   ──── get_result (service) ─────────────▶ returns the result when execute finishes
```

`example_interfaces/action/Fibonacci`: goal `int32 order`, feedback `int32[] sequence`, result `int32[] sequence`.
It stands in for any long job (homing an axis, moving to a pose) that reports progress and can be canceled.

**Launch files** start many processes with names, remaps and parameters in one command:

| `--ros-args` flag | Effect | Launch equivalent |
|---|---|---|
| `-r chatter:=/robot/heartbeat` | remap a topic name | `remappings=[...]` |
| `-r __node:=new_name` | rename the node | `name="new_name"` |
| `-p rate:=5.0` | set a parameter | `parameters=[{...}]` |
| `--params-file x.yaml` | many parameters | `parameters=["x.yaml"]` |

Packaged nodes use `launch_ros.actions.Node(package=..., executable=...)`; plain scripts (like ours) use `ExecuteProcess`.

In [ ]:
from roboprep import ros
ros.ros("ros2 interface show example_interfaces/action/Fibonacci")

## Task 1: the action server
Node `fib_server`, action `/fibonacci`. Reject goals with order outside 0..50. Each step: append, publish feedback,
sleep 0.1 s. Honor cancel requests between steps.

In [ ]:
%%writefile fib_server.py
import time
import rclpy
from example_interfaces.action import Fibonacci
from rclpy.action import ActionServer, CancelResponse, GoalResponse
from rclpy.callback_groups import ReentrantCallbackGroup
from rclpy.executors import ExternalShutdownException, MultiThreadedExecutor
from rclpy.node import Node


class FibServer(Node):
    def __init__(self):
        super().__init__("fib_server")
        self.server = ActionServer(self, Fibonacci, "fibonacci", execute_callback=self.execute,
                                   goal_callback=self.on_goal, cancel_callback=self.on_cancel,
                                   callback_group=ReentrantCallbackGroup())

    def on_goal(self, goal):
        return GoalResponse.ACCEPT  # TODO: reject orders outside 0..50

    def on_cancel(self, goal_handle):
        return CancelResponse.ACCEPT

    def execute(self, gh):
        seq = [0, 1]
        # TODO: loop, check gh.is_cancel_requested (then gh.canceled() and return), append,
        #       gh.publish_feedback(Fibonacci.Feedback(sequence=seq)), sleep 0.1 s
        gh.succeed()
        return Fibonacci.Result(sequence=seq)


def main():
    rclpy.init()
    node = FibServer()
    ex = MultiThreadedExecutor()     # lets the cancel callback run while execute() sleeps
    ex.add_node(node)
    try:
        ex.spin()
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

In [ ]:
%%writefile heartbeat.py
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from std_msgs.msg import String


class Heartbeat(Node):
    def __init__(self):
        super().__init__("heartbeat")
        rate = self.declare_parameter("rate", 1.0).value
        self.pub = self.create_publisher(String, "chatter", 10)
        self.create_timer(1.0 / rate, lambda: self.pub.publish(String(data="alive")))


def main():
    rclpy.init()
    try:
        rclpy.spin(Heartbeat())
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

## Task 2: one launch file for both
Start `fib_server.py`, and `heartbeat.py` with its `chatter` topic remapped to `/robot/heartbeat` and `rate` = 5.0.

In [ ]:
%%writefile bringup.launch.py
from launch import LaunchDescription
from launch.actions import ExecuteProcess


def generate_launch_description():
    return LaunchDescription([
        # TODO: two ExecuteProcess actions
    ])

In [ ]:
import time
ros.start("ros2 launch bringup.launch.py", log="launch.log")
time.sleep(4)
ros.ros('ros2 action send_goal /fibonacci example_interfaces/action/Fibonacci "{order: 5}" --feedback | tail -8', timeout=30)
ros.ros("timeout 3 ros2 topic hz /robot/heartbeat", timeout=10)
ros.stop_all()

In [ ]:
lab.check("action")
lab.check("launch")

In [ ]:
%%writefile fib_server.cpp
// C++ (rclcpp) version of Task 1 (optional, same checker). Grade it with lab.check("action", "fib_server.cpp"):
// the grader builds it with colcon as package rdl_fib_server, executable fib_server (depends on rclcpp, rclcpp_action, example_interfaces).
#include <chrono>
#include <thread>
#include <rclcpp/rclcpp.hpp>
#include <rclcpp_action/rclcpp_action.hpp>
#include <example_interfaces/action/fibonacci.hpp>
using Fibonacci = example_interfaces::action::Fibonacci;
using GoalHandle = rclcpp_action::ServerGoalHandle<Fibonacci>;
using namespace std::chrono_literals;

class FibServer : public rclcpp::Node {
public:
    FibServer() : Node("fib_server") {
        // TODO: rclcpp_action::create_server<Fibonacci>(this, "fibonacci", goal, cancel, accepted)
        //   goal: ACCEPT_AND_EXECUTE for order 0..50, else REJECT
        //   cancel: ACCEPT
        //   accepted: run execute(gh) on a new thread so the executor keeps serving cancel requests
    }
private:
    static void execute(const std::shared_ptr<GoalHandle> gh) {
        (void)gh;
        // TODO: build the sequence, publish feedback every 100 ms, honor is_canceling(), then succeed(result)
    }
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<FibServer>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("action", "fib_server.cpp")  # uncomment after writing the C++ cell above

## Checkpoint: Math
**M1 (Apply).** A goal with order 10 at 0.1 s per step takes how many seconds (to one decimal)?

**M2 (Apply).** Last element of the result for order 5?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Apply).** "Home the Z axis" can take up to 20 s and must be stoppable from a GUI. Interface?
- a) a service that returns when homing ends
- b) an action: accepted or rejected immediately, feedback during the move, cancelable, result at the end
- c) a topic the GUI publishes once

**K2 (Analyze).** Why does the server use a MultiThreadedExecutor with a reentrant callback group? Describe what would
happen to a cancel request with the default single-threaded spin.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros04.h`).
3. In `roboprep/checks/ros04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()